# Create the raw folder if it doesn't exist

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/")

True

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/")

True

# Move the uploaded files to our project's raw folder

In [0]:
files_to_move = ["beneficiaries.csv", "claims.csv", "providers.csv", "diagnosis_codes.csv", "procedure_codes.csv"]

for file in files_to_move:
    try:
        dbutils.fs.cp(f"/Volumes/workspace/default/data_pipeline_project_14_11_26/{file}", f"/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/{file}")
        print(f"Moved {file} to raw/ staging folder")
    except Exception as e:
        print(f"Could not move {file}. Please check if it uploaded correctly. Error: {e}")

Moved beneficiaries.csv to raw/ staging folder
Moved claims.csv to raw/ staging folder
Moved providers.csv to raw/ staging folder
Moved diagnosis_codes.csv to raw/ staging folder
Moved procedure_codes.csv to raw/ staging folder


Build the Bronze Layer (Raw to Delta)

In [0]:
from pyspark.sql.functions import current_timestamp, col
tables = ["beneficiaries", "claims", "providers", "diagnosis_codes", "procedure_codes"]

for i in tables:
    print(f"Ingesting {i}")

    # 1. Read the raw CSV file
    df = spark.read.csv(f"/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/{i}.csv", header=True)

    # 2. Add Medallion audit columns
    df_bronze = df.withColumn("Bronze_timestamp", current_timestamp()).withColumn("Bronze_file_path", col("_metadata.file_path"))

    # 3. Write to Bronze zone as Delta format
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/{i}")

print("All Bronze tables successfully written to Delta!")

Ingesting beneficiaries
Ingesting claims
Ingesting providers
Ingesting diagnosis_codes
Ingesting procedure_codes
All Bronze tables successfully written to Delta!


check the delta tables

In [0]:
df_bronze = spark.read.format("delta").load(f"/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/beneficiaries")

df_bronze_claims = spark.read.format("delta").load(f"/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/claims")

df_bronze.display()


beneficiary_id,birth_date,gender,city,state,postal_code,region,Bronze_timestamp,Bronze_file_path
B00001,1950-08-15,M,Kolkata,WB,689113,East,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00002,1972-08-21,F,Durgapur,WB,872737,East,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00003,1950-05-31,M,Asansol,WB,281322,East,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00004,1950-12-07,M,Delhi,DL,978059,North,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00005,1991-05-02,M,Mumbai,MH,745728,West,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00006,1994-07-02,M,Pune,MH,215302,West,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00007,1997-11-18,F,Bengaluru,KA,550316,South,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00008,1968-05-08,F,Chennai,TN,934087,South,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00009,1994-03-21,M,Hyderabad,TS,462172,South,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00010,1996-10-23,M,Bhubaneswar,OD,499072,East,2026-11-25T12:54:19.947Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
